In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from xgboost import XGBClassifier

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("../data/disease10k.csv")

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["has_disease"].value_counts())
print("\nClass percentages:")
print(df["has_disease"].value_counts(normalize=True))

# =========================
# 2. DEFINE FEATURES AND TARGET
# =========================
X = df.drop("has_disease", axis=1)
y = df["has_disease"]

# =========================
# 3. TRAIN / VALIDATION / TEST SPLIT
#    70% train, 15% validation, 15% test
#    Stratify keeps original class distribution
# =========================
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=123
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=123
)

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation class distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

# =========================
# 4. IDENTIFY NUMERIC AND CATEGORICAL COLUMNS
# =========================
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nNumeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# =========================
# 5. PREPROCESSING
#    Numeric: median imputation + scaling
#    Categorical: most frequent imputation + one-hot encoding
# =========================
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# =========================
# 6. COMPUTE SCALE_POS_WEIGHT
#    This penalizes missing positive class more
# =========================
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos

print("\nscale_pos_weight:", scale_pos_weight)

# =========================
# 7. BUILD XGBOOST PIPELINE
# =========================
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=123,
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    ))
])

# =========================
# 8. HYPERPARAMETER GRID
#    scoring='recall' because recall is priority
# =========================
param_grid_xgb = {
    "model__n_estimators": [100, 500, 1000],
    "model__max_depth": [3, 6, 9],
    "model__learning_rate": [0.01, 0.1, 0.2],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0],
    "model__gamma": [0, 0.1, 0.5] 
}

grid_xgb = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid_xgb,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

# =========================
# 9. FIT GRID SEARCH ON TRAINING DATA ONLY
# =========================
grid_xgb.fit(X_train, y_train)

print("\nBest XGBoost params:")
print(grid_xgb.best_params_)

print("\nBest XGBoost CV recall:")
print(grid_xgb.best_score_)

best_model = grid_xgb.best_estimator_

# =========================
# 10. GET VALIDATION PROBABILITIES
#    We tune threshold on validation set
# =========================
val_probs = best_model.predict_proba(X_val)[:, 1]

# =========================
# 11. THRESHOLD SEARCH
#    We do not trust default threshold 0.5
# =========================
thresholds = np.arange(0.05, 0.95, 0.01)

results = []

for threshold in thresholds:
    val_pred = (val_probs >= threshold).astype(int)

    recall = recall_score(y_val, val_pred, zero_division=0)
    precision = precision_score(y_val, val_pred, zero_division=0)
    f1 = f1_score(y_val, val_pred, zero_division=0)

    results.append({
        "threshold": threshold,
        "recall": recall,
        "precision": precision,
        "f1": f1
    })

threshold_df = pd.DataFrame(results)

print("\nThreshold tuning results (top 15):")
print(
    threshold_df.sort_values(
        by=["recall", "f1", "precision"],
        ascending=[False, False, False]
    ).head(15)
)

# =========================
# 12. THRESHOLD SELECTION RULE
#    Keep thresholds with recall >= 0.90
#    Among them choose best F1, then precision
# =========================
candidate_thresholds = threshold_df[threshold_df["recall"] >= 0.97]

if len(candidate_thresholds) > 0:
    best_row = candidate_thresholds.sort_values(
        by=["f1", "precision"],
        ascending=[False, False]
    ).iloc[0]
else:
    best_row = threshold_df.sort_values(
        by=["f1", "recall"],
        ascending=[False, False]
    ).iloc[0]

best_threshold = best_row["threshold"]

print("\nBEST THRESHOLD:")
print(best_row)


Dataset shape: (10000, 7)

Class distribution:
has_disease
0    9851
1     149
Name: count, dtype: int64

Class percentages:
has_disease
0    0.9851
1    0.0149
Name: proportion, dtype: float64

Shapes:
X_train: (7000, 6)
X_val  : (1500, 6)
X_test : (1500, 6)

Train class distribution:
has_disease
0    0.985143
1    0.014857
Name: proportion, dtype: float64

Validation class distribution:
has_disease
0    0.984667
1    0.015333
Name: proportion, dtype: float64

Test class distribution:
has_disease
0    0.985333
1    0.014667
Name: proportion, dtype: float64

Numeric columns: ['age', 'bmi', 'blood_pressure', 'glucose_level', 'family_history']
Categorical columns: ['physical_activity_level']

scale_pos_weight: 66.3076923076923
Fitting 5 folds for each of 729 candidates, totalling 3645 fits

Best XGBoost params:
{'model__colsample_bytree': 0.6, 'model__gamma': 0, 'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 100, 'model__subsample': 1.0}

Best XGBoost CV reca

In [13]:

# =========================
# 13. FINAL VALIDATION PREDICTIONS
# =========================
val_pred_final = (val_probs >= best_threshold).astype(int)

# =========================
# 14. VALIDATION METRICS
# =========================
val_cm = confusion_matrix(y_val, val_pred_final)
val_recall = recall_score(y_val, val_pred_final, zero_division=0)
val_precision = precision_score(y_val, val_pred_final, zero_division=0)
val_f1 = f1_score(y_val, val_pred_final, zero_division=0)
val_roc_auc = roc_auc_score(y_val, val_probs)
val_pr_auc = average_precision_score(y_val, val_probs)

print("\nVALIDATION CONFUSION MATRIX:")
print(val_cm)

print("\nFINAL VALIDATION METRICS:")
print("Recall:", val_recall)
print("Precision:", val_precision)
print("F1:", val_f1)
print("ROC-AUC:", val_roc_auc)
print("PR-AUC:", val_pr_auc)

print("\nValidation classification report:")
print(classification_report(y_val, val_pred_final, zero_division=0))



VALIDATION CONFUSION MATRIX:
[[876 601]
 [  0  23]]

FINAL VALIDATION METRICS:
Recall: 1.0
Precision: 0.03685897435897436
F1: 0.07109737248840804
ROC-AUC: 0.9308822230726207
PR-AUC: 0.2905222732162988

Validation classification report:
              precision    recall  f1-score   support

           0       1.00      0.59      0.74      1477
           1       0.04      1.00      0.07        23

    accuracy                           0.60      1500
   macro avg       0.52      0.80      0.41      1500
weighted avg       0.99      0.60      0.73      1500



In [14]:
# =========================
# TEST SET EVALUATION
# =========================
test_probs = best_model.predict_proba(X_test)[:, 1]
test_pred_final = (test_probs >= best_threshold).astype(int)

test_cm = confusion_matrix(y_test, test_pred_final)
test_recall = recall_score(y_test, test_pred_final, zero_division=0)
test_precision = precision_score(y_test, test_pred_final, zero_division=0)
test_f1 = f1_score(y_test, test_pred_final, zero_division=0)
test_roc_auc = roc_auc_score(y_test, test_probs)
test_pr_auc = average_precision_score(y_test, test_probs)

print("\nTEST CONFUSION MATRIX:")
print(test_cm)

print("\nFINAL TEST METRICS:")
print("Recall:", test_recall)
print("Precision:", test_precision)
print("F1:", test_f1)
print("ROC-AUC:", test_roc_auc)
print("PR-AUC:", test_pr_auc)

print("\nTest classification report:")
print(classification_report(y_test, test_pred_final, zero_division=0))


TEST CONFUSION MATRIX:
[[921 557]
 [  3  19]]

FINAL TEST METRICS:
Recall: 0.8636363636363636
Precision: 0.03298611111111111
F1: 0.06354515050167224
ROC-AUC: 0.854748431541395
PR-AUC: 0.21982960039920396

Test classification report:
              precision    recall  f1-score   support

           0       1.00      0.62      0.77      1478
           1       0.03      0.86      0.06        22

    accuracy                           0.63      1500
   macro avg       0.51      0.74      0.42      1500
weighted avg       0.98      0.63      0.76      1500

